In [ ]:
import osmium
import geopandas as gpd
from shapely.geometry import LineString
from tqdm import tqdm
import os

class RoadHandler(osmium.SimpleHandler):
    def __init__(self, pbar=None):
        super().__init__()
        self.data = []
        self.pbar = pbar

    def way(self, w):
        if "highway" in w.tags:
            coords = [(n.lon, n.lat) for n in w.nodes]
            self.data.append({
                "highway": w.tags.get("highway"),
                "geometry": LineString(coords)
            })
        if self.pbar:
            self.pbar.update(1)

# First pass: count number of ways in file (to size tqdm)
class WayCounter(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.count = 0
    def way(self, w):
        self.count += 1

counter = WayCounter()
counter.apply_file("united-kingdom-250910.osm.pbf")

with tqdm(total=counter.count, desc="Processing ways") as pbar:
    h = RoadHandler(pbar=pbar)
    h.apply_file("united-kingdom-250910.osm.pbf", locations=True)

roads = gpd.GeoDataFrame(h.data, crs="EPSG:4326")
roads.to_file("uk_roads.gpkg", layer="roads", driver="GPKG")
print(f"Saved {len(roads)} road segments")
